In [ ]:
# 1. Let WhisperX install itself and all its required background tools first
!pip install git+https://github.com/m-bain/whisperx.git

# 2. SLEDGEHAMMER: Forcefully overwrite the core PyTorch trio to match the Colab GPU
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import whisperx
import os
import json
import gc
import torch

# 1. Configuration for GPU
device = "cuda"
batch_size = 16
compute_type = "float16"

print("Loading model into GPU...")
model = whisperx.load_model("large-v2", device, compute_type=compute_type)

audio_folder = "/audio" #folder name
output_folder = "transcripts"
os.makedirs(output_folder, exist_ok=True)

# 2. Loop through every file and process it
for filename in os.listdir(audio_folder):
    if filename.endswith(".wav") or filename.endswith(".mp3"):
        print(f"\nProcessing {filename}...")
        audio_path = os.path.join(audio_folder, filename)

        # Load and transcribe
        audio = whisperx.load_audio(audio_path)
        result = model.transcribe(audio, batch_size=batch_size)

        # Save the result to a JSON file
        output_path = os.path.join(output_folder, f"{filename}.json")
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(result["segments"], f, ensure_ascii=False, indent=4)

        print(f"Finished {filename}! Saved to {output_path}")

        # Free up GPU memory between files just to be safe
        gc.collect()
        torch.cuda.empty_cache()

print("\nAll done! Check the 'transcripts' folder on the left.")

In [ ]:
import shutil
from google.colab import files

print("Zipping the transcripts folder...")
#creat zip file of output json
shutil.make_archive('transcripts_backup', 'zip', 'transcripts')

print("Triggering browser download...")
#browser to download it
files.download('transcripts_backup.zip')